# PG-MoE — 联合训练（ResNet3D vision expert 版）

用 Hang 训好的 **ResNet3D r3d_18** 作为 vision 端（standalone 89.3%），换掉之前的 Qwen 路线。

**Pipeline**（每个 batch）：

```
RGB frames (B, 3, 60, 112, 112) ──> ResNet3DExpert (frozen) ──> (B, T_v≈8, 256) ──┐
                                                                                    ├─ CrossAttn ─┐
IMU (B, 6, 192) ─> IMUExpert (pretrained 82.33%) ──> (B, 12, 256) ─────────────────┘             │
                                                                                                  ▼
                                                                                       GatedFusion (α weighted)
                                                                                                  │
IMU ──> PhaseArbitrator ──> α(t) (B, 12) ──────────────────────────────────────────────────────> classify
```

**关键决策：**
1. **预先把 RGB 跑过 ResNet3D 抽 token，缓存成小文件（~7 MB）**——一次性 10 分钟，之后训练就快
2. ResNet3D **冻结**，不更新它的权重（防 catastrophic forgetting；Hang 已经训到 89.3%）
3. IMU expert 用 pretrained，end-to-end 微调

**路径假设**（你 Drive 实际结构）：
- IMU files: `MyDrive/MMAI/utd_mhad/Inertial/`
- RGB videos: `MyDrive/MMAI/utd_mhad/RGB-part{1..4}/`
- IMU 权重: `MyDrive/MMAI/pgmoe_ckpt/imu_expert.pt`
- ResNet3D 权重: `MyDrive/MMAI/pgmoe_ckpt/resnet3d_vision_expert.pt`

---

## 1. 挂载 Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 路径 + 超参

注意 `CODE_DIR`——指向你 Drive 上 repo 实际位置。从截图看可能是 `Multi-Modal-AI/project/final/code/`，根据你实际改。

In [ ]:
import os, sys

# Drive 上 repo 路径（按你实际改）
CODE_DIR = "/content/drive/MyDrive/Multi-Modal-AI/project/final/code"

MMAI_ROOT     = "/content/drive/MyDrive/MMAI"
DATA_ROOT     = f"{MMAI_ROOT}/utd_mhad"
CKPT_DIR      = f"{MMAI_ROOT}/pgmoe_ckpt"
SAVE_DIR      = CKPT_DIR

IMU_CKPT      = f"{CKPT_DIR}/imu_expert.pt"
RESNET_CKPT   = f"{CKPT_DIR}/resnet3d_vision_expert.pt"
VTOKEN_CACHE  = f"{CKPT_DIR}/vision_tokens_resnet3d.pt"

# 训练超参
NUM_CLASSES      = 27
D_MODEL          = 256
T_I              = 12
TARGET_FRAMES    = 60
IMG_SIZE         = 112
EPOCHS           = 50
BATCH_SIZE       = 16
WEIGHT_DECAY     = 1e-4
DROPOUT          = 0.3
FOCAL_GAMMA      = 2.0

# v2: 差异学习率 (pretrained IMU 用低 lr, 其他随机初始化的用高 lr)
LR_IMU           = 1e-5    # 已经训到 82.33%, 别动太多
LR_VISION        = 1e-3    # 随机初始化, 需要快速学起来
LR_OTHER         = 5e-4    # cross-attn / phase-arbitrator / head

# v2: 删掉 entropy loss (它把 alpha 推向 0 造成 mode collapse)
# 换成 balance loss: 强制 alpha 整体均值靠近 0.5, 防止全押 IMU
ALPHA_VAR_WEIGHT = 0.5    # 加大: 鼓励 alpha 在时间步之间起伏
ALPHA_BAL_WEIGHT = 0.5    # 新: 全体 alpha 平均要接近 0.5

os.makedirs(SAVE_DIR, exist_ok=True)
sys.path.insert(0, CODE_DIR)

for label, p in [("CODE_DIR", CODE_DIR), ("Inertial", f"{DATA_ROOT}/Inertial"),
                  ("RGB-part1", f"{DATA_ROOT}/RGB-part1"),
                  ("IMU_CKPT", IMU_CKPT), ("RESNET_CKPT", RESNET_CKPT)]:
    print(f"  {'✓' if os.path.exists(p) else '✗'} {label}: {p}")

## 3. 导入

In [ ]:
import gc, time
import numpy as np
import cv2
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

from models.pgmoe import PGMoE, FocalLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 4. 定义 ResNet3DExpert（Hang 的架构）

直接复制 Hang 在 `MMAI_Final.ipynb` cell 23 里的 `ResNet3DExpert` 类。我们只用 `return_tokens=True` 那个分支拿 token，不用他的分类头。

In [ ]:
class ResNet3DExpert(nn.Module):
    def __init__(self, d_model=256, num_classes=27, dropout=0.3):
        super().__init__()
        backbone = torchvision.models.video.r3d_18(
            weights=torchvision.models.video.R3D_18_Weights.DEFAULT)
        self.features = nn.Sequential(
            backbone.stem, backbone.layer1, backbone.layer2,
            backbone.layer3, backbone.layer4,
        )
        self.spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1))
        self.proj = nn.Sequential(
            nn.Linear(512, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, 65, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8, dim_feedforward=d_model * 4,
            dropout=dropout, activation='gelu', batch_first=True)
        self.temporal_transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

    def forward(self, x, return_tokens=False):
        B = x.shape[0]
        feat = self.features(x)
        feat = self.spatial_pool(feat).squeeze(-1).squeeze(-1).permute(0, 2, 1)
        T_out = feat.shape[1]
        feat = self.proj(feat)
        cls = self.cls_token.expand(B, -1, -1)
        feat = torch.cat([cls, feat], dim=1)
        feat = feat + self.pos_embed[:, :T_out + 1, :]
        feat = self.temporal_transformer(feat)
        if return_tokens:
            return feat[:, 1:, :]
        return self.head(feat[:, 0, :])

# 加载 Hang 的权重
print("Loading ResNet3D vision expert weights...")
ckpt = torch.load(RESNET_CKPT, map_location='cpu', weights_only=False)
print("  ckpt keys:", list(ckpt.keys()) if isinstance(ckpt, dict) else "(state_dict)")

resnet3d = ResNet3DExpert(d_model=D_MODEL, num_classes=NUM_CLASSES, dropout=DROPOUT).to(device)
if isinstance(ckpt, dict) and "model_state" in ckpt:
    resnet3d.load_state_dict(ckpt["model_state"])
    if "best_acc" in ckpt:
        print(f"  Hang 报告 best_acc = {ckpt['best_acc']:.4f}")
else:
    resnet3d.load_state_dict(ckpt)
resnet3d.eval()
for p in resnet3d.parameters():
    p.requires_grad = False
print(f"  loaded, frozen, params = {sum(p.numel() for p in resnet3d.parameters()):,}")

## 5. 预计算 vision tokens（一次性，缓存到 Drive）

如果 cache 文件已存在，跳过；否则遍历所有视频，每个 video 跑一次 ResNet3D，存成 `(T_v, 256)` 小 tensor。

860 video × 8 token × 256 dim ≈ **6.7 MB**——比 Qwen cache 的 13.5 GB 小 2000 倍。

A100 上一遍大约 5–10 分钟。

In [ ]:
def find_video(action, subject, trial):
    fname = f"a{action}_s{subject}_t{trial}_color.avi"
    for part in ["RGB-part1", "RGB-part2", "RGB-part3", "RGB-part4"]:
        p = os.path.join(DATA_ROOT, part, fname)
        if os.path.exists(p):
            return p
    return None

def extract_frames(video_path, imu_path=None, n_frames=TARGET_FRAMES, img_size=IMG_SIZE):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    start_ratio, end_ratio = 0.0, 1.0
    # IMU-aligned crop (same trick Hang used)
    if imu_path and os.path.exists(imu_path):
        imu = sio.loadmat(imu_path)["d_iner"]
        mag = np.sqrt((imu[:, :3] ** 2).sum(axis=1))
        thr = mag.mean() + 0.3 * mag.std()
        active = np.where(mag > thr)[0]
        if len(active) > 0:
            T = len(mag)
            start_ratio = max(0.0, (active[0] - 15) / T)
            end_ratio = min(1.0, (active[-1] + 15) / T)
    f0 = int(total * start_ratio)
    f1 = max(int(total * end_ratio), f0 + n_frames)
    indices = np.linspace(f0, min(f1, total - 1), n_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (img_size, img_size))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    cap.release()
    if len(frames) < n_frames:
        return None
    return np.array(frames, np.float32) / 255.0   # (60, 112, 112, 3)


if os.path.exists(VTOKEN_CACHE):
    print(f"Loading existing token cache: {VTOKEN_CACHE}")
    vision_cache = torch.load(VTOKEN_CACHE, map_location="cpu", weights_only=False)
    print(f"  loaded {len(vision_cache)} samples")
else:
    print(f"Extracting vision tokens from videos (one-time, ~5-10 min on A100)...")
    vision_cache = {}
    success, fail = 0, 0
    t0 = time.time()
    for action in range(1, NUM_CLASSES + 1):
        for subject in range(1, 9):
            for trial in range(1, 5):
                vp = find_video(action, subject, trial)
                if vp is None:
                    fail += 1; continue
                ip = os.path.join(DATA_ROOT, "Inertial",
                                  f"a{action}_s{subject}_t{trial}_inertial.mat")
                frames = extract_frames(vp, ip)
                if frames is None:
                    fail += 1; continue
                # to torch: (1, 3, 60, 112, 112)
                x = torch.from_numpy(frames).permute(3, 0, 1, 2).unsqueeze(0).to(device)
                with torch.no_grad():
                    tokens = resnet3d(x, return_tokens=True).squeeze(0).cpu()  # (T_v, 256)
                vision_cache[(action, subject, trial)] = tokens.to(torch.float16)
                success += 1
        print(f"  action {action:2d}/27 done  (success={success} fail={fail}  t={time.time()-t0:.0f}s)")
    torch.save(vision_cache, VTOKEN_CACHE)
    print(f"\n  saved {len(vision_cache)} samples to {VTOKEN_CACHE}")
    print(f"  total time {time.time()-t0:.1f}s")

# 看一个 sample 的 T_v
sample = next(iter(vision_cache.values()))
T_V_ACTUAL = sample.shape[0]
print(f"\nResNet3D output T_v = {T_V_ACTUAL} tokens per sample (each {sample.shape[1]}-d)")

## 6. 释放 ResNet3D 显存

Token 已经全部缓存，ResNet3D 不再需要。删掉它，腾出 GPU 显存给 PG-MoE 训练。

In [ ]:
del resnet3d
torch.cuda.empty_cache()
gc.collect()
print("ResNet3D removed.")

## 7. 联合 Dataset

返回 `(vision_tokens, imu, label)`：
- vision_tokens: `(T_v≈8, 256)`（已经 spatial+temporal encoded 的 token）
- imu: `(6, 192)`
- label: 0–26

Train/test split：subjects {1,3,5,7} vs {2,4,6,8}——和 IMU expert 完全一致。

In [ ]:
IMU_LEN = 192
TRAIN_SUBJECTS = {1, 3, 5, 7}
TEST_SUBJECTS  = {2, 4, 6, 8}

def load_imu(action, subject, trial):
    fname = f"a{action}_s{subject}_t{trial}_inertial.mat"
    fpath = os.path.join(DATA_ROOT, "Inertial", fname)
    if not os.path.exists(fpath):
        return None
    data = sio.loadmat(fpath)["d_iner"].astype(np.float32)
    if data.shape[0] < IMU_LEN:
        pad = np.zeros((IMU_LEN - data.shape[0], 6), np.float32)
        data = np.concatenate([data, pad], axis=0)
    return torch.from_numpy(data[:IMU_LEN]).T.contiguous()


class JointDataset(Dataset):
    def __init__(self, vision_cache, train=True):
        allowed = TRAIN_SUBJECTS if train else TEST_SUBJECTS
        self.samples = []
        for (a, s, t), v_tok in vision_cache.items():
            if s not in allowed:
                continue
            imu = load_imu(a, s, t)
            if imu is None:
                continue
            self.samples.append((v_tok, imu, a - 1))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        v, i, y = self.samples[idx]
        return v.float(), i, y

train_ds = JointDataset(vision_cache, train=True)
test_ds  = JointDataset(vision_cache, train=False)
print(f"Train: {len(train_ds)}  |  Test: {len(test_ds)}")
v0, i0, y0 = train_ds[0]
print(f"Sample 0: vision {tuple(v0.shape)}, imu {tuple(i0.shape)}, label {y0}")

## 8. 构造 PG-MoE + 加载 IMU 预训练

我们的 `PGMoE` 类自带 `VisionProjector`——把进来的 vision token 再过一遍 `LayerNorm + Linear(d_in→256) + temporal transformer`。**注意 `vision_d_in=256` 而不是默认 2048**，因为 ResNet3D 已经 project 到 256 了。

In [ ]:
model = PGMoE(
    num_classes=NUM_CLASSES,
    d_model=D_MODEL,
    T_i=T_I,
    vision_n_frames=T_V_ACTUAL,      # ResNet3D 的实际 T_v
    vision_d_in=D_MODEL,             # 256 (ResNet3D 输出已经是 d_model)
    vision_n_layers=2,
    attn_n_layers=2,
    attn_n_heads=4,
    dropout=DROPOUT,
).to(device)

model.load_pretrained_imu(IMU_CKPT, device=device)
print("Loaded pretrained IMU encoder from", IMU_CKPT)

n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Params: {n_total:,} total ({n_trainable:,} trainable)")

## 9. Forward smoke test

In [ ]:
tmp_loader = DataLoader(test_ds, batch_size=4, shuffle=False)
v, i, y = next(iter(tmp_loader))
v, i = v.to(device), i.to(device)
model.eval()
with torch.no_grad():
    logits, alpha = model(v, i, return_alpha=True)
print(f"Input  vision: {tuple(v.shape)}")
print(f"Input  imu   : {tuple(i.shape)}")
print(f"Output logits: {tuple(logits.shape)}")
print(f"Output alpha : {tuple(alpha.shape)}")
print(f"Alpha range  : {float(alpha.min()):.3f} to {float(alpha.max()):.3f}")

## 10. 训练 setup (v2 改动)

**v1 结果反思：** α(t) 全贴 0.05–0.18，模型基本只用 IMU。原因：
1. IMU pretrained 82.33%，vision projector 随机初始化 → 梯度倾向 IMU
2. Entropy loss 推 α 远离 0.5，但选了向 0 的方向（不是向 1）
3. α ≈ 0 → vision 梯度被 0 衰减 → vision 永远学不动

**v2 两个改动：**

### A. 差异学习率
- IMU 用 1e-5（小，别破坏 pretrained）
- Vision 用 1e-3（大，让它快速 catch up）
- 其他用 5e-4

### B. 把 entropy loss 换成 balance loss
- 旧：`entropy = H(α)` → 推 α 远离 0.5，但方向不定
- 新：`balance = (α.mean() - 0.5)²` → **强制全体 α 平均靠近 0.5**，禁止全押一边
- 同时加大 variance loss 权重 0.1→0.5


In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

# 差异学习率: 给 pretrained IMU 一个小 lr, vision/其他用大 lr
optimizer = torch.optim.AdamW([
    {"params": model.imu_expert.parameters(),        "lr": LR_IMU},
    {"params": model.vision_proj.parameters(),       "lr": LR_VISION},
    {"params": model.cross_attn.parameters(),        "lr": LR_OTHER},
    {"params": model.phase_arbitrator.parameters(),  "lr": LR_OTHER},
    {"params": model.fusion.parameters(),            "lr": LR_OTHER},
    {"params": model.norm.parameters(),              "lr": LR_OTHER},
    {"params": model.head.parameters(),              "lr": LR_OTHER},
], weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = FocalLoss(gamma=FOCAL_GAMMA)

for i, g in enumerate(optimizer.param_groups):
    n = sum(p.numel() for p in g['params'])
    print(f"  group {i}: lr={g['lr']:.1e}, params={n:,}")
print(f"\nTrain batches: {len(train_loader)}, Test batches: {len(test_loader)}")

## 11. 评估函数

In [ ]:
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for v, i, y in loader:
            logits = model(v.to(device), i.to(device))
            preds.extend(logits.argmax(1).cpu().numpy())
            labels.extend(y.numpy())
    return accuracy_score(labels, preds), np.array(preds), np.array(labels)

## 12. 训练 v2 (差异 LR + balance loss, 不再用 entropy)

**v2 总 loss**:
```
loss = focal_cls
     + 0.5 * (- alpha.var(dim=1).mean())     # 鼓励时间方差 (per-sample)
     + 0.5 * (alpha.mean() - 0.5).pow(2)     # 鼓励整体均值≈0.5 (反对全押一边)
```

**预期：**
- 看 `bal` 这一列：第一次跑应该从 ~0.16 (alpha 平均 ~0.1) 降到 ~0
- 看 `var` 列：应该越训越负，表示 α 沿时间分化
- 看 `acc`: 目标 >91% (上一版 89.77%, ResNet3D baseline 89.3%)
- A100 上 50 epoch 仍是 ~5-10 分钟


In [ ]:
best_acc, best_epoch = 0.0, -1
history = {"cls": [], "var": [], "bal": [], "acc": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    sum_cls = sum_var = sum_bal = 0.0
    n_seen = 0
    for v, i, y in train_loader:
        v, i, y = v.to(device), i.to(device), y.to(device)
        optimizer.zero_grad()
        logits, alpha = model(v, i, return_alpha=True)

        cls_loss = criterion(logits, y)
        # 鼓励 alpha 在 12 个时间步之间起伏 (per-sample variance)
        var_loss = -alpha.var(dim=1).mean()
        # 鼓励全体 alpha 平均接近 0.5 (反对全押 IMU 或全押 vision)
        bal_loss = (alpha.mean() - 0.5).pow(2)

        loss = cls_loss + ALPHA_VAR_WEIGHT * var_loss + ALPHA_BAL_WEIGHT * bal_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        bs = v.size(0)
        sum_cls += cls_loss.item() * bs
        sum_var += var_loss.item() * bs
        sum_bal += bal_loss.item() * bs
        n_seen += bs
    scheduler.step()

    avg_cls, avg_var, avg_bal = sum_cls/n_seen, sum_var/n_seen, sum_bal/n_seen
    acc, _, _ = evaluate(model, test_loader)
    history["cls"].append(avg_cls); history["var"].append(avg_var)
    history["bal"].append(avg_bal); history["acc"].append(acc)

    flag = ""
    if acc > best_acc:
        best_acc, best_epoch = acc, epoch
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "pgmoe_best.pt"))
        flag = "  *"
    print(f"Epoch {epoch:3d} | cls {avg_cls:.4f} | var {avg_var:+.4f} | bal {avg_bal:.4f} "
          f"| acc {acc:.4f} | {time.time()-t0:.1f}s{flag}")

print(f"\nBest test acc: {best_acc:.4f} ({best_acc*100:.2f}%) at epoch {best_epoch}")
print(f"  baselines: IMU 82.33% | ResNet3D 89.3% | v1 PG-MoE 89.77% | Oracle 98.4%")

## 13. 训练曲线（4 panel）

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].plot(history["cls"]); axes[0].set_title("Focal cls loss"); axes[0].set_xlabel("epoch"); axes[0].grid(True, alpha=0.3)
axes[1].plot(history["var"], color="C1"); axes[1].set_title("α var loss (more negative = more time variance)"); axes[1].set_xlabel("epoch"); axes[1].grid(True, alpha=0.3)
axes[2].plot(history["bal"], color="C2"); axes[2].set_title("α balance loss (lower = mean(α) closer to 0.5)"); axes[2].set_xlabel("epoch"); axes[2].grid(True, alpha=0.3)
axes[3].plot(history["acc"]); axes[3].set_title("Test accuracy"); axes[3].set_xlabel("epoch"); axes[3].grid(True, alpha=0.3)
axes[3].axhline(0.8233, ls="--", color="red",    alpha=0.6, label="IMU 82.33%")
axes[3].axhline(0.893,  ls="--", color="green",  alpha=0.6, label="ResNet3D 89.3%")
axes[3].axhline(0.8977, ls=":",  color="orange", alpha=0.6, label="v1 PG-MoE 89.77%")
axes[3].legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "pgmoe_training_curve.png"), dpi=120)
plt.show()

## 14. 最佳模型评估 + per-class

对比 IMU 单模态最差的 3 类：
- c4  right arm throw  (IMU 25%)
- c18 right hand knock (IMU 44%)
- c16 tennis serve     (IMU 50%)

PG-MoE 应该在这些类上明显比 IMU 高。

In [ ]:
UTD_LABELS = [
    "swipe left", "swipe right", "wave", "clap", "throw",
    "arm cross", "basketball shoot", "draw x",
    "draw circle CW", "draw circle CCW", "draw triangle",
    "bowling", "boxing", "baseball swing", "tennis swing",
    "arm curl", "tennis serve", "two hand push", "knock",
    "catch", "pickup and throw", "jogging", "walking",
    "sit to stand", "stand to sit", "forward lunge", "squat",
]
IMU_PER_CLASS = {4: 0.250, 18: 0.438, 16: 0.500}

model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "pgmoe_best.pt"),
                                  map_location=device, weights_only=False))
acc, preds, labels = evaluate(model, test_loader)
print(f"PG-MoE test acc: {acc:.4f} ({acc*100:.2f}%)\n")

rows = []
for c in range(NUM_CLASSES):
    mask = labels == c
    if mask.sum() == 0: continue
    rows.append((c, float((preds[mask] == c).mean()), int(mask.sum())))
rows.sort(key=lambda r: r[1])
print("Per-class accuracy (worst first):")
for c, ac, n in rows:
    bar = "#" * int(ac * 30)
    base = f" (IMU was {IMU_PER_CLASS[c]:.3f})" if c in IMU_PER_CLASS else ""
    print(f"  c{c:2d} acc {ac:.3f} n={n:2d} | {bar:<30} {UTD_LABELS[c]}{base}")

cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap="Blues")
plt.colorbar(); plt.title(f"PG-MoE Confusion Matrix (acc={acc*100:.2f}%)")
plt.xlabel("predicted"); plt.ylabel("true")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "pgmoe_confusion_matrix.png"), dpi=120)
plt.show()

## 15. 训练后 α(t) 可视化

Throw / walking / squat 三个动作画 α(t)：
- throw 应该在 impact 时刻 α 升高（信 vision）
- walking / squat α 整体低（信 IMU）
- 不再像未训练时三条线重合在 0.527

In [ ]:
def get_alpha_for(action_1idx, subject=2, trial=1):
    key = (action_1idx, subject, trial)
    if key not in vision_cache:
        return None
    v = vision_cache[key].float().unsqueeze(0).to(device)
    imu = load_imu(action_1idx, subject, trial)
    if imu is None: return None
    imu = imu.unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        _, alpha = model(v, imu, return_alpha=True)
    return alpha.squeeze().cpu().numpy()

demo = [(5, "throw", "red"), (23, "walking", "blue"), (27, "squat", "green")]
fig, ax = plt.subplots(figsize=(10, 4))
for aid, lbl, col in demo:
    a = get_alpha_for(aid, subject=2, trial=1)
    if a is None: continue
    ax.plot(np.linspace(0, 1, len(a)), a, "o-", color=col, label=lbl, linewidth=2)
ax.axhline(0.5, ls="--", color="gray", alpha=0.5)
ax.set_ylim(0, 1)
ax.set_xlabel("normalized time within action")
ax.set_ylabel("α (1=trust vision, 0=trust IMU)")
ax.set_title("α(t) after PG-MoE joint training")
ax.grid(True, alpha=0.3); ax.legend(loc="best")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "alpha_trained.png"), dpi=120)
plt.show()

## 16. 打包下载

zip 内容：pgmoe_best.pt + 3 张 png + results.txt。下载完解压塞进 repo `figures/` 和 `checkpoints/`。

In [ ]:
import shutil
BUNDLE_DIR = "/content/pgmoe_archive"
os.makedirs(BUNDLE_DIR, exist_ok=True)
for src in ["pgmoe_best.pt", "pgmoe_training_curve.png",
            "pgmoe_confusion_matrix.png", "alpha_trained.png"]:
    p = os.path.join(SAVE_DIR, src)
    if os.path.exists(p):
        shutil.copy(p, os.path.join(BUNDLE_DIR, src))

lines = [
    "# PG-MoE (ResNet3D vision) Joint Training Results\n\n",
    f"Best test accuracy: {best_acc:.4f}  ({best_acc*100:.2f}%)\n",
    f"Best epoch:         {best_epoch}\n\n",
    "## Baselines\n",
    "  IMU only:      82.33%\n",
    "  Vision (Qwen):  58.6%  (not used)\n",
    "  Vision (ResNet3D): 89.3%  (used as vision expert)\n",
    "  Oracle ceiling: 98.4%\n\n",
    "## Per-class accuracy (sorted ascending)\n",
]
for c, ac, n in rows:
    base = f"  (IMU was {IMU_PER_CLASS[c]:.3f})" if c in IMU_PER_CLASS else ""
    lines.append(f"  c{c:2d}  acc={ac:.3f}  n={n:2d}  {UTD_LABELS[c]}{base}\n")
with open(os.path.join(BUNDLE_DIR, "pgmoe_results.txt"), "w") as f:
    f.writelines(lines)

print("Bundle:")
for fn in sorted(os.listdir(BUNDLE_DIR)):
    sz = os.path.getsize(os.path.join(BUNDLE_DIR, fn)) / (1024*1024)
    print(f"  {fn}  ({sz:.2f} MB)")

zip_path = "/content/pgmoe_archive.zip"
shutil.make_archive(zip_path.replace(".zip", ""), "zip", BUNDLE_DIR)
print(f"\nzipped: {zip_path}  ({os.path.getsize(zip_path)/(1024*1024):.2f} MB)")

from google.colab import files
files.download(zip_path)

## 完事

跑完发我：
1. Best test acc
2. α(t) 图三条线长什么样（throw 有没有峰）
3. per-class 里 c4/c18/c16 涨了多少